In [1]:
import torch
import torch.nn.functional as F
 
# ======================================================================
# 1. LOAD THE DATASET
# ======================================================================
words = open('names.txt', 'r').read().splitlines()
 
# build the character <-> integer mappings ('.' is the start/end token = 0)
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)          # 27

In [2]:
block_size = 3                  # context length: how many chars we use to predict the next
 
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)   # current context  ->  X
            Y.append(ix)        # next character    ->  Y
            context = context[1:] + [ix]   # slide the window
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [3]:
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
 
Xtr,  Ytr  = build_dataset(words[:n1])     # 80% train
Xdev, Ydev = build_dataset(words[n1:n2])   # 10% dev/val
Xte,  Yte  = build_dataset(words[n2:])     # 10% test

In [4]:
# ======================================================================
# 2. INITIALIZE THE NETWORK  (MLP + BatchNorm)
# ======================================================================
n_emb    = 10    # size of each character embedding vector
n_hidden = 200   # neurons in the hidden layer
 
g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_emb),            generator=g)
W1 = torch.randn((n_emb * block_size, n_hidden), generator=g) * (5/3) / ((n_emb * block_size) ** 0.5)  # Kaiming init (tanh gain 5/3)
# NOTE: there is no b1 — BatchNorm's bnbias makes a bias on this layer redundant.
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01   # tiny -> logits start near 0 -> loss starts ~ln(27)=3.30
b2 = torch.randn(vocab_size, generator=g) * 0
 

In [5]:
# --- BatchNorm: LEARNED parameters (scale & shift) ---
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
 
# --- BatchNorm: RUNNING stats (NOT learned) ---
# updated with an exponential moving average during training,
# then used at eval/sample time when there is no batch to normalize over.
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running  = torch.ones((1, n_hidden))
 
parameters = [C, W1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters), 'parameters')
for p in parameters:
    p.requires_grad = True

12097 parameters


In [7]:
for i in range(max_steps):
    # --- minibatch ---
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]
 
    # --- forward pass ---
    emb     = C[Xb]                       # (B, block_size, n_emb)
    embcat  = emb.view(emb.shape[0], -1)  # (B, block_size*n_emb)  flatten the context
    hpreact = embcat @ W1                  # (B, n_hidden)  -- no bias term
 
    # --- BatchNorm ---
    bnmeani = hpreact.mean(0, keepdim=True)    # keepdim=True -> shape (1, n_hidden) so it broadcasts
    bnstdi  = hpreact.std(0,  keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    with torch.no_grad():                      # update running stats; no gradients here
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
        bnstd_running  = 0.999 * bnstd_running  + 0.001 * bnstdi
    # -----------------
 
    h      = torch.tanh(hpreact)          # (B, n_hidden)
    logits = h @ W2 + b2                    # (B, vocab_size)
    loss   = F.cross_entropy(logits, Yb)
 
    # --- backward pass ---
    for p in parameters:
        p.grad = None                       # zero the grads BEFORE backward
    loss.backward()
 
    # --- update (SGD with a simple step decay) ---
    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad
 
    # --- track ---
    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())
 

      0/ 200000: 3.3121
  10000/ 200000: 2.1756
  20000/ 200000: 2.4320
  30000/ 200000: 2.1226
  40000/ 200000: 2.1291
  50000/ 200000: 2.0928
  60000/ 200000: 2.3900
  70000/ 200000: 2.3196
  80000/ 200000: 2.2756
  90000/ 200000: 1.9173
 100000/ 200000: 2.1991
 110000/ 200000: 2.2357
 120000/ 200000: 2.0928
 130000/ 200000: 1.7492
 140000/ 200000: 1.9435
 150000/ 200000: 1.9660
 160000/ 200000: 1.9354
 170000/ 200000: 1.7196
 180000/ 200000: 2.0396
 190000/ 200000: 2.1193


In [8]:
# 4. EVALUATE LOSS ON THE SPLITS  (uses the RUNNING stats, not batch stats)
# ======================================================================
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (Xtr,  Ytr),
        'val':   (Xdev, Ydev),
        'test':  (Xte,  Yte),
    }[split]
    emb     = C[x]
    embcat  = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    h       = torch.tanh(hpreact)
    logits  = h @ W2 + b2
    loss    = F.cross_entropy(logits, y)
    print(split, loss.item())
 
split_loss('train')
split_loss('val')
 

train 2.065643548965454
val 2.1062614917755127


In [9]:
# 5. SAMPLE NEW NAMES  (also uses the running stats)
# ======================================================================
g2 = torch.Generator().manual_seed(2147483647 + 10)
for _ in range(20):
    out = []
    context = [0] * block_size
    while True:
        emb     = C[torch.tensor([context])]
        embcat  = emb.view(emb.shape[0], -1)
        hpreact = embcat @ W1
        hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
        h       = torch.tanh(hpreact)
        logits  = h @ W2 + b2
        probs   = F.softmax(logits, dim=1)
        ix      = torch.multinomial(probs, num_samples=1, generator=g2).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    print(''.join(itos[i] for i in out))

montalmyah.
see.
med.
rylle.
emman.
endraege.
zered.
elin.
shi.
jen.
eden.
estanaraelyn.
malail.
noshub.
rishirael.
kindreelle.
xiterian.
breyce.
ruyah.
faeh.
